# Tutorial 10 — Promote and serve the model

**Goal.** Move the artifact from Tutorial 09 behind a typed local HTTP service, exercise valid and invalid requests, and compare online scores with offline predictions.

**Prerequisites.** Complete Tutorial 09; install `poetry install -E ml -E serving`. Docker is optional.

**Produces.** A model metadata record, health response, successful score, safe validation errors, warm latency measurements, and an offline/online parity check.


In [ ]:
import json
import time
from pathlib import Path

from examples.model_service.app import ModelRuntime

artifact_dir = Path("runs/tutorial-09/evaluation")
artifacts = sorted(artifact_dir.glob("models/*.joblib"))
if not artifacts:
    raise FileNotFoundError("Run Tutorial 09 first; no .joblib artifact found")
artifact = artifacts[0]


runtime = ModelRuntime(artifact)
print(runtime.metadata)

## Start the reference service

The service loads the model once per process. The request includes an event identifier, a timezone-aware prediction timestamp, and the same feature names used during training.


In [ ]:
from examples.model_service.app import create_app
from fastapi.testclient import TestClient

client = TestClient(create_app(artifact))
print(client.get("/health").json())
valid = {
    "event_id": "evt-tutorial-10",
    "prediction_timestamp": "2026-01-08T12:00:00Z",
    "features": {"amount": 120.0, "hour": 12.0},
}
response = client.post("/score", json=valid)
print(response.status_code, response.json())
bad = client.post("/score", json={**valid, "features": {"unknown_feature": 1}})
print("invalid request:", bad.status_code, bad.json())
assert response.status_code == 200 and bad.status_code == 422

## Measure local hand-off quality

These are local measurements, not production guarantees. Warm p50/p95 latency, artifact size, validation failures, and confidence fallback should be tracked alongside model quality when deciding whether to promote.


In [ ]:
durations = []
for i in range(50):
    start = time.perf_counter()
    score = client.post("/score", json={**valid, "event_id": f"evt-{i}"})
    durations.append((time.perf_counter() - start) * 1000)
latencies = sorted(durations)
p50, p95 = latencies[24], latencies[47]
body = response.json()
decision = "human_review" if body["fraud_score"] < 0.6 else "approve_or_decline"
print(
    {
        "p50_ms": p50,
        "p95_ms": p95,
        "artifact_bytes": artifact.stat().st_size,
        "confidence_action": decision,
    }
)

## Replay and verify parity

Replay a small historical slice through `/score`, then compare its scores with the saved offline prediction rows. A mismatch usually means feature ordering, timestamp handling, or model-version metadata changed during promotion.


In [ ]:
from fraudtwin.ml.baseline import score_model_artifact

offline = score_model_artifact(artifact, [valid["features"]])[0]
online = response.json()["fraud_score"]
assert abs(offline - online) < 1e-9
print(
    json.dumps(
        {"model_version": body["model_id"], "offline": offline, "online": online, "parity": "ok"},
        indent=2,
    )
)

Run the containerized variant with `docker build -t fraudtwin-model examples/model_service` and `docker run -p 8000:8000 -e MODEL_ARTIFACT=/models/model.joblib fraudtwin-model`. Next: [Tutorial 11 — detect data, domain, and concept shift](11-checkpoint-resume-scale.ipynb) and the [production serving guide](../production-serving.md).
